In [1]:
cd ..

/home/drkareemkamal/Documents/cancer-survival-predictor


# 1. Data Preparation

In [ ]:
# split the data and fix mutation paths 
!python main.py --stage data 

In [2]:
!python main.py --stage features --force

weave: Logged in as Weights & Biases user: dr-kareem-kamal.
weave: View Weave data at https://wandb.ai/dr-kareem-kamal/cancer-survival-predictor/weave

[run clinical] python -m src.features.clinical --config configs/data.yaml
weave: 🍩 https://wandb.ai/dr-kareem-kamal/cancer-survival-predictor/r/call/019e1366-b65e-70fb-8bad-2f1ed8d0160e
clinical features: (8459, 61)  patients: 8459
wrote /home/drkareemkamal/Documents/cancer-survival-predictor/data/processed/features/clinical.parquet

[run expression] python -m src.features.expression --config configs/data.yaml --top-k 5000
resolved RNA-Seq files: 8459 / 8459
pass 1/2: streaming variance on 5919 train TSVs
pass1 variance: 100%|███████████████████████| 5919/5919 [09:33<00:00, 10.32it/s]
  computed variance for 60660 genes
  selected top-5000 most variable genes
pass 2/2: extracting top-5000 genes from all 8459 patients
pass2 extract: 100%|████████████████████████| 8459/8459 [14:40<00:00,  9.61it/s]
  matrix shape: (8459, 5000)  RAM: 169 M

In [3]:
# let's verfiy the result
import pandas as pd
for f in ['clinical','expression','mutation']:
    df = pd.read_parquet(f'data/processed/features/{f}.parquet')
    print(f, df.shape, df['TCGA_Barcode'].nunique(), 'patients')

clinical (8459, 61) 8459 patients
expression (8459, 5001) 8459 patients
mutation (8459, 1005) 8459 patients


# 2. Pathology LLM fine-tune

## 2.1 Build pathological QA 

In [ ]:
!python main.py --stage pathology-qa

## 2.2 Chain of Thought using GPT-4o-mini

In [ ]:
!python -m src.training.distill_cot \
       --config configs/pathology_llm.yaml \
       --train-qa data/processed/pathology/qa_train.jsonl \
       --out data/processed/pathology/qa_train_cot.jsonl

## 2.3 Finetune Pathological text using `Unsloth/Qwen2.5-7b` 

In [ ]:
!python main.py --stage pathology-train

weave: Logged in as Weights & Biases user: dr-kareem-kamal.
weave: View Weave data at https://wandb.ai/dr-kareem-kamal/cancer-survival-predictor/weave

[run pathology-train] python -m src.training.unsloth_finetune --config configs/pathology_llm.yaml
weave: 🍩 https://wandb.ai/dr-kareem-kamal/cancer-survival-predictor/r/call/019e032d-4d2f-7b40-88be-90b7745e9274
weave: Logged in as Weights & Biases user: dr-kareem-kamal.
weave: View Weave data at https://wandb.ai/dr-kareem-kamal/cancer-survival-predictor/weave
[weave] tracking 'pathology-train' under project 'cancer-survival-predictor'
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
weave: 🍩 https://wandb.ai/dr-kareem-kamal/cancer-survival-predictor/r/call/019e032d-667b-73ef-b80f-ef07cfc49378
[weave.trace.weave_client|INFO]🍩 https://wandb.ai/dr-kareem-kamal/cancer-survival-predictor/r/call/019e032d-667b-73ef-b80f-ef07cfc49378
🦥 Unsloth Zoo will now patch everything to make training faster!
will push final adapter 

## 2.4 Extract pathology features for downstream multimodal training

In [ ]:
!python -m src.training.extract_features \
    --config configs/pathology_llm.yaml \
    --adapter-dir models/PathQwen2.5/final

weave: Logged in as Weights & Biases user: dr-kareem-kamal.
weave: View Weave data at https://wandb.ai/dr-kareem-kamal/cancer-survival-predictor/weave
[weave] tracking 'pathology-extract' under project 'cancer-survival-predictor'
weave: 🍩 https://wandb.ai/dr-kareem-kamal/cancer-survival-predictor/r/call/019e1383-feb7-72e4-8417-2da16f39c11a
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
cohort: 8459 reports
loading unsloth/Qwen2.5-7B-Instruct-bnb-4bit + adapter models/PathQwen2.5/final
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignor

## 2.5 Score Path-Qwen2.5 on the held-out test set

In [ ]:
!python -m src.evaluation.pathology_eval \
    --struct data/processed/features/pathology_struct.parquet \
    --qa-test data/processed/pathology/qa_test.jsonl \
    --output models/PathQwen2.5/eval_pathology.json

# 3. Survival models 

## 3.1 Baselines

In [ ]:
## Cox PH on clinical only (sanity baseline)
!python main.py --stage survival-baselines

In [ ]:
# Add Modalities 
!python -m src.models.baselines --modalities clinical,expression,mutation,pathology_struct,pathology_embed

## 3.2 Deep multimodal models

In [ ]:
# Autoencoder 
!python -m src.models.train_multimodal --config configs/multimodal.yaml --model autoencoder


In [ ]:
# Transformer fusion (
!python -m src.models.train_multimodal --config configs/multimodal.yaml --model transformer

In [ ]:
# Ensemble (autoencoder + transformer + meta-learner)
!python -m src.models.train_multimodal --config configs/multimodal.yaml --model ensemble